# uxlens × LocateAnything — real CRO audit (free T4)

Run a **real** uxlens audit with NVIDIA's open-vocabulary vision model, on a free Colab T4.

## How to run (do this in order — no restart needed)
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Wait for the model download (~6 GB, first run only)

That's it. Because this is a fresh runtime, everything installs *before* any import, so no session restart is required.

> ⚠️ LocateAnything-3B is **non-commercial** (research/personal use). uxlens itself is MIT.

## 1. Memory + quantization settings (must run first)

In [ ]:
import os
# Reduce GPU memory fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Load the model in 4-bit so the 3B fits a free T4 (~2.5 GB vs ~6 GB).
os.environ["UXLENS_LOAD_4BIT"] = "1"
print("settings applied")

## 2. Install uxlens + browser + model deps (pinned)

In [ ]:
# Pinned commit => always the latest fixes, never a cached old build.
# uxlens[model] pulls torch, transformers==4.57.1, bitsandbytes, decord, lmdb.
!pip install -q "uxlens[model] @ git+https://github.com/kannajune/uxlint.git@cea9328"
!playwright install chromium
!playwright install-deps 2>/dev/null || true
print("install done")

## 3. Sanity checks: GPU + fresh code

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Runtime to T4!")

import uxlens.locator.locate_anything as la, inspect
src = inspect.getsource(la.LocateAnythingLocator)
print("has use_cache fix:", "use_cache=True" in src)
print("has 4-bit support:", "load_in_4bit" in src)

## 4. Audit your URL
First run downloads the model (~6 GB) — give it a few minutes.

In [ ]:
URL = "https://www.fixurdevice.in/"   # <-- your site
VIEWPORT = "mobile"                    # or "desktop"

from uxlens.audit import audit
from uxlens.report import print_summary, write_annotated_image

result = audit(URL, out_dir="out", viewport=VIEWPORT, backend="locate-anything")
print_summary(result)
write_annotated_image(result, "out/annotated.png")

## 5. See the annotated screenshot

In [ ]:
from IPython.display import Image, display
display(Image('out/annotated.png'))

## 6. (If boxes look wrong) raw model output
Share this output and we'll tune the parser.

In [ ]:
from PIL import Image as PILImage
from uxlens.locator import get_locator

loc = get_locator('locate-anything')
img = PILImage.open('out/screenshot.png').convert('RGB')
raw = loc._raw_inference(loc._downscale(img), loc._prompt('primary call-to-action button'))
print(repr(raw))